**Objective 1**

U-Net Training Pipeline (PyTorch)

This notebook walks through the full supervised learning pipeline:

1. Load and inspect a 200-sample satellite image dataset
2. Build a custom PyTorch *Dataset* and *DataLoader*
3. Construct a U-Net with a ResNet-18 encoder
4. Train with BCE loss + Adam optimiser
5. Evaluate on a held-out test set
6. Run inference on a new image




step 1: import libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

step 2: custom dataset class

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir   = image_dir
        self.mask_dir    = mask_dir
        self.image_files = sorted([f for f in os.listdir(image_dir)
                                   if f.endswith('.JPG') or f.endswith('.PNG')])
        self.mask_files  = sorted([f for f in os.listdir(mask_dir)
                                   if f.endswith('.JPG') or f.endswith('.PNG')])
        self.transform   = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir,  self.mask_files[idx])
        image = Image.open(img_path).convert('RGB')
        mask  = Image.open(mask_path).convert('L')
        if self.transform:
            image = self.transform(image)
            mask  = self.transform(mask)
        return image, mask

step 3: transforms, dataloaders & dataset verification

In [ ]:
# ── Paths — update these to your local data folders ──────────────────
image_dir = 'data/images'
mask_dir  = 'data/masks'

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

dataset = CustomDataset(image_dir=image_dir, mask_dir=mask_dir, transform=transform)
print(f'Number of images found : {len(dataset)}')
print(f'Number of masks found  : {len(dataset)}')
print(f'Total samples          : {len(dataset)}')

train_set, test_set = train_test_split(dataset, test_size=0.1, random_state=42)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False)
print(f'Train: {len(train_set)} | Test: {len(test_set)}')

step 4: visualise a batch of images and masks

In [ ]:
def imshow(img, title=None):
    img   = img / 2 + 0.5          # un-normalise
    npimg = img.numpy()
    plt.figure(figsize=(14, 4))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title, fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

dataiter        = iter(train_loader)
images, masks   = next(dataiter)
imshow(vutils.make_grid(images), title='Satellite Images — Training Batch')
imshow(vutils.make_grid(masks),  title='Corresponding Masks')

step 5: U-Net architecture (ResNet-18 Encoder)

In [ ]:
class UNet(nn.Module):
    def __init__(self, num_classes=1):
        super(UNet, self).__init__()
        resnet        = models.resnet18(pretrained=True)
        self.encoder  = nn.Sequential(*list(resnet.children())[:-2])
        self.upconv1  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.upconv2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.upconv3  = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.upconv4  = nn.ConvTranspose2d( 64,  32, 2, stride=2)
        self.conv_final = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        x = self.encoder(x)
        x = self.upconv1(x)
        x = self.upconv2(x)
        x = self.upconv3(x)
        x = self.upconv4(x)
        x = self.conv_final(x)
        return torch.sigmoid(x)

model = UNet(num_classes=1).to(device)
print(model)

step 6: loss function & optimiser

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


step 7: train the model

In [ ]:
def train_model(model, criterion, optimizer, train_loader, num_epochs=25):
    model.train()
    train_losses = []
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, masks in train_loader:
            inputs, masks = inputs.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            masks   = transforms.Resize(outputs.shape[2:])(masks)
            loss    = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        print(f'Epoch {epoch}/{num_epochs - 1}, Loss: {epoch_loss:.4f}')
    return train_losses

train_losses = train_model(model, criterion, optimizer, train_loader, num_epochs=25)


step 8: plot training loss curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.tight_layout()
plt.savefig('results/training_loss_curve.png', dpi=150)
plt.show()

step 9: evaluate on test set

In [ ]:
def test_model(model, test_loader):
    model.eval()
    all_preds, all_masks = [], []
    with torch.no_grad():
        for inputs, masks in test_loader:
            inputs, masks = inputs.to(device), masks.to(device)
            outputs = model(inputs)
            masks   = transforms.Resize(outputs.shape[2:])(masks)
            preds   = (outputs > 0.5).float()
            all_preds.extend(preds.cpu().numpy().flatten())
            all_masks.extend(masks.cpu().numpy().flatten())
    return np.array(all_preds), np.array(all_masks)

test_preds, test_masks = test_model(model, test_loader)

step 10: confusion matrix & accuracy

In [ ]:
test_preds_flat = (test_preds > 0.5).astype(int)
test_masks_flat = (test_masks > 0.5).astype(int)

conf_matrix = confusion_matrix(test_masks_flat, test_preds_flat)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Background','Building'],
            yticklabels=['Background','Building'])
plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()

accuracy = accuracy_score(test_masks_flat, test_preds_flat)
print(f'Accuracy: {accuracy:.4f}')

step 11: save model & run inference on a new image

In [ ]:
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/unet_building.pth')
print('Model saved!')

def inference(model, image_path, transform):
    model.eval()
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'Image not found at {image_path}')
    image             = Image.open(image_path).convert('RGB')
    image_transformed = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image_transformed)
        pred   = (output > 0.5).float()
    return image, pred.cpu().squeeze().numpy()

# ── Update the path below to an actual image ─────────────────────────
image_path = 'data/images/triple_0.JPG'

original_image, pred_mask = inference(model, image_path, transform)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original_image); axes[0].set_title('Input Satellite Image'); axes[0].axis('off')
axes[1].imshow(pred_mask, cmap='gray'); axes[1].set_title('Predicted Building Mask'); axes[1].axis('off')
plt.suptitle('Automatic Building Detection — U-Net Prediction', fontsize=13)
plt.tight_layout()
plt.savefig('results/prediction.png', dpi=150)
plt.show()